**Konteks:** Opsional: memasang MediaPipe, OpenCV, dan mengunduh model `pose_landmarker_full.task`. Hanya perlu dijalankan bila ekstraksi pose (Bagian 2) diulang.

In [ ]:
# OPSIONAL - hanya untuk Bagian 2 (ekstraksi pose ulang)
!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.30 opencv-python-headless pandas numpy tqdm
!wget -q -O pose_landmarker_full.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!ls -lh pose_landmarker_full.task

**Konteks:** Menghubungkan Google Drive ke Colab agar data Figshare dan semua keluaran dapat dibaca/ditulis.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

**Konteks:** Mendefinisikan seluruh path data dan folder keluaran Week 2 (`05. Week 2`) serta saklar eksekusi. Data asli Figshare hanya dibaca, tidak diubah.

In [ ]:
# Path data dan keluaran, saklar, cek isi folder
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

# lokasi di Google Drive
DRIVE        = Path("/content/drive/MyDrive/KP-TA MCI")
FIGSHARE_DIR = DRIVE / "02. Figshare"            # data asli (tidak diubah)
DATA_ROOT    = str(FIGSHARE_DIR / "Videos")      # 73 video
IMU_ROOT     = FIGSHARE_DIR / "IMU"              # file IMU 128 Hz
BASE         = DRIVE / "05. Week 2"              # SEMUA keluaran disimpan di sini
WEEK1_DIR    = DRIVE / "03. Week 1"              # hanya cadangan (lihat OUT_CSV_LAMA)

# turunan dari BASE
KEYPOINT_DIR    = BASE / "keypoints_roi"         # pose mentah per video (.npz)
ROI_CSV         = BASE / "roi_manual.csv"        # kotak ROI manual per video
OUT_CSV_ROI     = BASE / "keluaran1_roi_manual.csv"
OUT_BANDING     = BASE / "perbandingan_lama_vs_roi.csv"
RESAMPLED_DIR   = BASE / "resampled_32hz"        # pose + IMU + label pada 32 Hz
FITUR_DIR       = BASE / "fitur_li_v2"           # fitur gaya Li dkk.
POSE_GRID_DIR   = BASE / "pose_grid32"           # pose 17 sendi untuk ST-GCN
HASIL_FOGBENCH  = BASE / "hasil_fogbench"        # berkas per fold + kartu metode
KODE_DIR        = BASE / "kode_model"            # salinan models/*.py
OUT_CSV_LAMA    = BASE / "keluaran_1.csv"        # hasil ekstraksi full-frame (lama)
if not OUT_CSV_LAMA.exists() and (WEEK1_DIR / "keluaran_1.csv").exists():
    OUT_CSV_LAMA = WEEK1_DIR / "keluaran_1.csv"  # belum dipindah: pakai yang di Week 1
MODEL_PATH      = "pose_landmarker_full.task"    # model MediaPipe (diunduh di Sel 0.1)

for d in (BASE, RESAMPLED_DIR, FITUR_DIR, POSE_GRID_DIR, HASIL_FOGBENCH, KODE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# saklar
RUN_ROI_MANUAL            = False   # True = gambar ROI manual (interaktif)
RUN_EKSTRAKSI_POSE        = False   # True = ekstraksi ulang pose (berjam-jam)
RUN_DIAGNOSTIK_VISUAL     = False   # True = tampilkan crop+skeleton video bermasalah
RUN_STGCN_3SEED_DAN_SKEMA = True    # True = jalankan Sel 7.5

# pemeriksaan
def _cek(label, path, harap=None):
    path = Path(path)
    if not path.exists():
        print(f"[KOSONG] {label}: {path}"); return
    n = len(list(path.glob('*'))) if path.is_dir() else None
    info = f" ({n} berkas)" if n is not None else ""
    ok = "OK" if (harap is None or n == harap) else f"CEK (harap {harap})"
    print(f"[{ok}] {label}: {path}{info}")

_cek("Videos", DATA_ROOT, 73); _cek("IMU", IMU_ROOT)
_cek("roi_manual.csv", ROI_CSV); _cek("keypoints_roi", KEYPOINT_DIR, 73)
_cek("resampled_32hz", RESAMPLED_DIR, 71); _cek("fitur_li_v2", FITUR_DIR)
_cek("pose_grid32", POSE_GRID_DIR, 71); _cek("keluaran_1.csv (lama)", OUT_CSV_LAMA)

**Konteks:** Fungsi untuk mencari file video berdasarkan nama rekaman dan mendaftar semua video. Hasilnya `remaining`, daftar 73 rekaman.

In [ ]:
# Cari file video per nama rekaman
VIDEO_EXTS = [".mp4", ".avi", ".mov", ".mkv", ".m4v"]

def find_video_path(root, stem):
    root = Path(root)
    for ext in VIDEO_EXTS:
        matches = list(root.rglob(f"{stem}{ext}"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Video {stem} tidak ditemukan di {root}")

def list_semua_video(root):
    root = Path(root)
    stems = set()
    for ext in VIDEO_EXTS:
        for p in root.rglob(f"*{ext}"):
            stems.add(p.stem)
    return sorted(stems)

remaining = list_semua_video(DATA_ROOT)
print(f"Total video ditemukan: {len(remaining)}")

**Konteks:** Fungsi penyimpanan hasil kualitas pose per rekaman ke CSV secara resumable (rekaman yang sudah ada dilewati) beserta baris median akhir.

In [ ]:
"""Simpan hasil per rekaman ke CSV secara resumable."""
import pandas as pd

COLUMNS = [
    "Rekaman", "Durasi (detik)", "Frame dengan pose terdeteksi (%)",
    "Frame dengan kedua pergelangan kaki terdeteksi (%)",
    "Confidence rerata", "Catatan",
]

def load_done_names(out_csv: Path) -> set[str]:
    """Nama rekaman yang sudah ada di CSV (dilewati)."""
    if not out_csv.exists():
        return set()
    df = pd.read_csv(out_csv)
    df = df[df["Rekaman"] != "Median seluruh rekaman"]
    return set(df["Rekaman"].astype(str))

def append_row(out_csv: Path, row: dict) -> None:
    """Tulis 1 baris hasil langsung ke CSV."""
    df_row = pd.DataFrame([row], columns=COLUMNS)
    write_header = not out_csv.exists()
    df_row.to_csv(out_csv, mode="a", header=write_header, index=False)

def finalize_with_median(out_csv: Path) -> pd.DataFrame:
    """Tambah baris median seluruh rekaman di akhir CSV."""
    df = pd.read_csv(out_csv)
    df = df[df["Rekaman"] != "Median seluruh rekaman"].copy()
    median_row = {
        "Rekaman": "Median seluruh rekaman",
        "Durasi (detik)": df["Durasi (detik)"].median(),
        "Frame dengan pose terdeteksi (%)": df["Frame dengan pose terdeteksi (%)"].median(),
        "Frame dengan kedua pergelangan kaki terdeteksi (%)": df[
            "Frame dengan kedua pergelangan kaki terdeteksi (%)"
        ].median(),
        "Confidence rerata": df["Confidence rerata"].median(),
        "Catatan": "",
    }
    df_out = pd.concat([df, pd.DataFrame([median_row])], ignore_index=True)
    df_out.to_csv(out_csv, index=False)
    return df_out

**Konteks:** Mendefinisikan `VideoStats` (penghitung frame terdeteksi dan visibility) dan `make_landmarker` untuk membuat detektor MediaPipe BlazePose.

In [ ]:
"""VideoStats + make_landmarker (dari keluaran1_pose_quality.py)."""
from dataclasses import dataclass, field
import numpy as np
try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision
except ImportError:
    mp = mp_tasks = vision = None
    print('[info] MediaPipe belum terpasang: tidak masalah kecuali Bagian 2 dijalankan ulang (Sel 0.1).')
import cv2

LEFT_ANKLE = 27
RIGHT_ANKLE = 28
VIS_THRESH = 0.5
BBOX_JUMP_THRESH = 0.25

@dataclass
class VideoStats:
    name: str
    fps: float = 30.0
    n_frames: int = 0
    n_pose_detected: int = 0
    n_ankles_detected: int = 0
    vis_sum: float = 0.0
    vis_count: int = 0
    bbox_centers: list = field(default_factory=list)

    @property
    def duration_s(self) -> float:
        return self.n_frames / self.fps if self.fps else 0.0

    @property
    def pct_pose(self) -> float:
        return 100.0 * self.n_pose_detected / self.n_frames if self.n_frames else 0.0

    @property
    def pct_ankles(self) -> float:
        return 100.0 * self.n_ankles_detected / self.n_frames if self.n_frames else 0.0

    @property
    def mean_confidence(self) -> float:
        return self.vis_sum / self.vis_count if self.vis_count else float("nan")

    def note(self) -> str:
        if self.n_frames == 0:
            return "video tidak terbaca"
        notes = []
        if self.pct_pose < 50:
            notes.append("deteksi pose rendah (<50% frame)")
        if self.pct_ankles < 50:
            notes.append("deteksi pergelangan kaki rendah (<50% frame)")
        if len(self.bbox_centers) >= 2:
            centers = np.array(self.bbox_centers)
            jumps = np.linalg.norm(np.diff(centers, axis=0), axis=1)
            if jumps.size and (jumps > BBOX_JUMP_THRESH).sum() / jumps.size > 0.05:
                notes.append("diduga ID switching pasien-pemeriksa (>5% lompatan bbox besar)")
        return "; ".join(notes) if notes else "aman"

    def to_row(self) -> dict:
        conf = self.mean_confidence
        return {
            "Rekaman": self.name,
            "Durasi (detik)": round(self.duration_s, 2),
            "Frame dengan pose terdeteksi (%)": round(self.pct_pose, 2),
            "Frame dengan kedua pergelangan kaki terdeteksi (%)": round(self.pct_ankles, 2),
            "Confidence rerata": round(conf, 4) if not np.isnan(conf) else None,
            "Catatan": self.note(),
        }


def make_landmarker(model_path: str, min_conf: float = 0.5):
    base_options = mp_tasks.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=min_conf,
        min_pose_presence_confidence=min_conf,
        min_tracking_confidence=min_conf,
    )
    return vision.PoseLandmarker.create_from_options(options)

**Konteks:** Antarmuka kanvas Colab untuk menggambar ROI pasien secara manual pada satu frame video, sehingga pasien terpisah dari pemeriksa.

In [ ]:
# Pilih ROI manual lewat canvas di Colab
import cv2
import base64
import numpy as np
from IPython.display import display, Javascript
from google.colab.output import eval_js

def _frame_ke_base64(path_video, posisi_relatif=0.3):
    """Frame di 30% durasi (pasien sudah siap) -> JPEG base64."""
    cap = cv2.VideoCapture(path_video)
    total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = int(total_frame * posisi_relatif)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Gagal baca frame dari {path_video}")
    h, w = frame.shape[:2]
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
    b64 = base64.b64encode(buf).decode("utf-8")
    return b64, w, h


def ambil_roi_manual(path_video, lebar_tampil=800):
    """Klik-seret ROI di canvas; hasil dalam piksel asli video."""
    b64, w_asli, h_asli = _frame_ke_base64(path_video)
    skala = lebar_tampil / w_asli
    h_tampil = int(h_asli * skala)

    js = Javascript(f"""
    async function ambilROI() {{
        const img = new Image();
        img.src = "data:image/jpeg;base64,{b64}";
        await img.decode();

        const canvas = document.createElement('canvas');
        canvas.width = {lebar_tampil};
        canvas.height = {h_tampil};
        canvas.style.cursor = 'crosshair';
        const ctx = canvas.getContext('2d');
        ctx.drawImage(img, 0, 0, {lebar_tampil}, {h_tampil});

        document.body.appendChild(canvas);
        const info = document.createElement('div');
        info.innerText = 'Klik dan seret untuk membuat kotak ROI di sekitar PASIEN, lalu lepas mouse.';
        document.body.appendChild(info);

        let x1, y1, x2, y2, drawing = false;

        canvas.addEventListener('mousedown', (e) => {{
            const r = canvas.getBoundingClientRect();
            x1 = e.clientX - r.left; y1 = e.clientY - r.top;
            drawing = true;
        }});
        canvas.addEventListener('mousemove', (e) => {{
            if (!drawing) return;
            const r = canvas.getBoundingClientRect();
            const cx = e.clientX - r.left, cy = e.clientY - r.top;
            ctx.drawImage(img, 0, 0, {lebar_tampil}, {h_tampil});
            ctx.strokeStyle = 'red'; ctx.lineWidth = 2;
            ctx.strokeRect(x1, y1, cx - x1, cy - y1);
        }});

        return new Promise((resolve) => {{
            canvas.addEventListener('mouseup', (e) => {{
                const r = canvas.getBoundingClientRect();
                x2 = e.clientX - r.left; y2 = e.clientY - r.top;
                drawing = false;
                canvas.remove(); info.remove();
                resolve([Math.min(x1,x2), Math.min(y1,y2), Math.max(x1,x2), Math.max(y1,y2)]);
            }});
        }});
    }}
    ambilROI();
    """)
    display(js)
    x1, y1, x2, y2 = eval_js("ambilROI()")

    # konversi dari koordinat canvas (tampilan) balik ke koordinat piksel asli
    x1, y1, x2, y2 = x1/skala, y1/skala, x2/skala, y2/skala
    return int(x1), int(y1), int(x2), int(y2), w_asli, h_asli

**Konteks:** Fungsi memuat/menyimpan ROI ke `roi_manual.csv` dan `crop_adaptif`, yang memotong frame ke ROI ditambah padding 15%.

In [ ]:
# Muat/simpan ROI dan crop adaptif
import pandas as pd
from pathlib import Path

# ROI_CSV didefinisikan di Sel 0.3 (05. Week 2/roi_manual.csv)
KOLOM_ROI = ["Rekaman", "x1", "y1", "x2", "y2", "frame_w", "frame_h"]

def load_roi_done():
    if ROI_CSV.exists():
        return pd.read_csv(ROI_CSV)
    return pd.DataFrame(columns=KOLOM_ROI)

def simpan_roi(nama_rekaman, x1, y1, x2, y2, w, h):
    df = load_roi_done()
    df = df[df["Rekaman"] != nama_rekaman]
    baris_baru = pd.DataFrame([{
        "Rekaman": nama_rekaman, "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        "frame_w": w, "frame_h": h,
    }])
    df = pd.concat([df, baris_baru], ignore_index=True)
    df.to_csv(ROI_CSV, index=False)
    print(f"[tersimpan] {nama_rekaman}: ROI ({x1},{y1})-({x2},{y2}) dari frame {w}x{h}")

PADDING_FRAC = 0.15  # padding = 15% dari ukuran ROI itu sendiri, bukan piksel tetap

def crop_adaptif(frame, x1, y1, x2, y2, frame_w, frame_h):
    """Crop ke ROI + padding 15% dari ukuran ROI."""
    lebar_roi = x2 - x1
    tinggi_roi = y2 - y1
    pad_x = int(lebar_roi * PADDING_FRAC)
    pad_y = int(tinggi_roi * PADDING_FRAC)

    x1p = max(0, int(x1 - pad_x))
    y1p = max(0, int(y1 - pad_y))
    x2p = min(int(frame_w), int(x2 + pad_x))
    y2p = min(int(frame_h), int(y2 + pad_y))
    return frame[y1p:y2p, x1p:x2p]

**Konteks:** Menjalankan pemilihan ROI hanya untuk video yang belum punya ROI. Bila `RUN_ROI_MANUAL=False`, memakai `roi_manual.csv` yang sudah ada.

In [ ]:
# Pilih ROI untuk video yang belum punya ROI
roi_df = load_roi_done()
sudah_selesai = set(roi_df["Rekaman"])
print(f"Total video: {len(remaining)} | Sudah ada ROI: {len(sudah_selesai)} | Sisa: {len(remaining) - len(sudah_selesai)}")

if RUN_ROI_MANUAL:
    for nama_rekaman in remaining:
        if nama_rekaman in sudah_selesai:
            continue
        path_video = find_video_path(DATA_ROOT, nama_rekaman)
        print(f"--- {nama_rekaman} ---")
        x1, y1, x2, y2, w, h = ambil_roi_manual(path_video)
        simpan_roi(nama_rekaman, x1, y1, x2, y2, w, h)
    roi_df = load_roi_done()
else:
    print("RUN_ROI_MANUAL=False -> tidak menggambar ROI (memakai roi_manual.csv yang ada).")

**Konteks:** Ekstraksi pose di dalam ROI untuk setiap video, lalu menyimpan keypoint, visibility, dan waktu ke file `.npz` serta statistik kualitasnya ke CSV.

In [ ]:
"""Ekstraksi pose di dalam ROI + simpan keypoint dan waktu ke .npz."""
import numpy as np

# KEYPOINT_DIR didefinisikan di Sel 0.3
def process_video_dengan_roi_v2(path: Path, roi_row, model_path: str, min_conf: float = 0.5) -> VideoStats:
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    stats = VideoStats(name=path.stem, fps=fps)

    x1, y1, x2, y2 = roi_row.x1, roi_row.y1, roi_row.x2, roi_row.y2
    w, h = roi_row.frame_w, roi_row.frame_h

    landmarker = make_landmarker(model_path, min_conf)
    frame_idx = 0
    t_list, xy_list, vis_list = [], [], []
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            stats.n_frames += 1
            frame_crop = crop_adaptif(frame, x1, y1, x2, y2, w, h)
            rgb = cv2.cvtColor(frame_crop, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(frame_idx * 1000 / fps)
            frame_idx += 1

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if not result.pose_landmarks:
                # tetap catat waktunya, isi NaN -- supaya nanti saat resampling
                # kita tahu ada "lubang" deteksi di titik waktu ini, bisa diinterpolasi
                t_list.append(timestamp_ms)
                xy_list.append(np.full((33, 2), np.nan))
                vis_list.append(np.full(33, np.nan))
                continue

            stats.n_pose_detected += 1
            lm = result.pose_landmarks[0]
            vis_vals = [p.visibility for p in lm]
            stats.vis_sum += float(np.mean(vis_vals))
            stats.vis_count += 1
            if lm[LEFT_ANKLE].visibility >= VIS_THRESH and lm[RIGHT_ANKLE].visibility >= VIS_THRESH:
                stats.n_ankles_detected += 1
            xs = [p.x for p in lm]; ys = [p.y for p in lm]
            stats.bbox_centers.append((float(np.mean(xs)), float(np.mean(ys))))

            t_list.append(timestamp_ms)
            xy_list.append(np.array([[p.x, p.y] for p in lm]))
            vis_list.append(np.array(vis_vals))
    finally:
        landmarker.close(); cap.release()

    np.savez(
        KEYPOINT_DIR / f"{path.stem}.npz",
        t_ms=np.array(t_list, dtype=np.float64),
        xy=np.array(xy_list, dtype=np.float32),    # bentuk (n_frame, 33, 2), koordinat 0-1 relatif ke ROI
        vis=np.array(vis_list, dtype=np.float32),  # bentuk (n_frame, 33)
    )
    return stats



def load_done_npz(keypoint_dir):
    return {p.stem for p in Path(keypoint_dir).glob("*.npz")}

done_kp = load_done_npz(KEYPOINT_DIR)
roi_df = load_roi_done()
target_kp = [r for r in roi_df["Rekaman"] if r not in done_kp]
print(f"Keypoint tersimpan: {len(done_kp)} | akan diproses: {len(target_kp)}")

if RUN_EKSTRAKSI_POSE:
    done_stat = load_done_names(OUT_CSV_ROI)
    for nama_rekaman in tqdm(target_kp, desc="Ekstraksi + simpan keypoint"):
        row = roi_df[roi_df["Rekaman"] == nama_rekaman].iloc[0]
        path_video = find_video_path(DATA_ROOT, nama_rekaman)
        st = process_video_dengan_roi_v2(path_video, row, MODEL_PATH)
        if nama_rekaman not in done_stat:
            append_row(OUT_CSV_ROI, st.to_row())
    df_final = finalize_with_median(OUT_CSV_ROI)
    print(f"\nSelesai. Hasil: {OUT_CSV_ROI}")
else:
    print("RUN_EKSTRAKSI_POSE=False -> memakai keypoints_roi/ yang sudah ada.")

**Konteks:** Membandingkan kualitas pose hasil lama (full-frame) dengan hasil ROI, dipasangkan per rekaman dan diuji dengan Wilcoxon. Hanya berjalan bila kedua CSV tersedia.

In [ ]:
if OUT_CSV_LAMA.exists() and OUT_CSV_ROI.exists():

    """Kualitas pose full-frame vs ROI, dipasangkan per rekaman."""
    import pandas as pd
    from pathlib import Path
    from scipy.stats import wilcoxon

    OUT_CSV_BARU = OUT_CSV_ROI

    def muat_bersih(path_csv):
        """Baca CSV tanpa baris median."""
        df = pd.read_csv(path_csv)
        return df[df["Rekaman"] != "Median seluruh rekaman"].copy()

    df_lama = muat_bersih(OUT_CSV_LAMA)
    df_baru = muat_bersih(OUT_CSV_BARU)

    # gabungkan berdasarkan nama rekaman -> satu baris per video, kolom lama & baru berdampingan
    gabung = df_lama.merge(
        df_baru, on="Rekaman", suffixes=(" (lama)", " (ROI)")
    )

    print(f"Video di hasil lama : {len(df_lama)}")
    print(f"Video di hasil ROI  : {len(df_baru)}")
    print(f"Video yang match    : {len(gabung)}")
    if len(gabung) < len(df_lama):
        hilang = set(df_lama["Rekaman"]) - set(gabung["Rekaman"])
        print(f"[peringatan] {len(hilang)} video tidak match: {sorted(hilang)[:5]}...")

    # ringkasan median, sebelum vs sesudah
    kolom_metrik = [
        "Frame dengan pose terdeteksi (%)",
        "Frame dengan kedua pergelangan kaki terdeteksi (%)",
        "Confidence rerata",
    ]

    print("\n=== Median seluruh rekaman: LAMA (full-frame) vs BARU (ROI manual) ===")
    for kolom in kolom_metrik:
        med_lama = gabung[f"{kolom} (lama)"].median()
        med_baru = gabung[f"{kolom} (ROI)"].median()
        delta = med_baru - med_lama
        print(f"{kolom:55s}  lama={med_lama:6.2f}  roi={med_baru:6.2f}  delta={delta:+6.2f}")

    # uji statistik: apakah kenaikannya signifikan, bukan cuma kebetulan?
    print("\n=== Uji Wilcoxon berpasangan (lama vs ROI, per rekaman) ===")
    for kolom in kolom_metrik:
        a = gabung[f"{kolom} (lama)"].dropna()
        b = gabung[f"{kolom} (ROI)"].dropna()
        idx_sama = a.index.intersection(b.index)
        a, b = a.loc[idx_sama], b.loc[idx_sama]
        if (a == b).all():
            print(f"{kolom:55s}  semua nilai identik, uji dilewati")
            continue
        stat, p = wilcoxon(a, b)
        lebih_baik = (b > a).sum()
        lebih_buruk = (b < a).sum()
        sama = (b == a).sum()
        print(f"{kolom:55s}  p={p:.4g}  (naik:{lebih_baik}, turun:{lebih_buruk}, sama:{sama})")

    # cek rekaman yang justru memburuk, perlu dicek manual
    gabung["delta_pose"] = (
        gabung["Frame dengan pose terdeteksi (%) (ROI)"] - gabung["Frame dengan pose terdeteksi (%) (lama)"]
    )
    memburuk = gabung[gabung["delta_pose"] < -5].sort_values("delta_pose")
    print(f"\n=== Rekaman yang justru memburuk >5 poin persen ({len(memburuk)} video) ===")
    if len(memburuk):
        print(memburuk[["Rekaman", "Frame dengan pose terdeteksi (%) (lama)",
                         "Frame dengan pose terdeteksi (%) (ROI)", "delta_pose"]].to_string(index=False))
    else:
        print("Tidak ada. Semua rekaman stabil atau membaik.")

    # simpan tabel gabungan untuk lampiran laporan
    gabung.to_csv(OUT_BANDING, index=False)
    print(f"\nTabel perbandingan lengkap disimpan ke: {OUT_BANDING}")
else:
    print('[lewati] keluaran_1.csv (lama) atau keluaran1_roi_manual.csv tidak ditemukan; sel ini opsional.')

**Konteks:** Diagnosis visual untuk rekaman yang memburuk: menampilkan frame cuplikan beserta hasil deteksi, guna melihat apakah pasien keluar ROI atau MediaPipe gagal.

In [ ]:
"""Cek visual video yang memburuk: pasien keluar ROI atau MediaPipe gagal?"""
from google.colab.patches import cv2_imshow

def diagnosa_visual_roi(nama_rekaman, titik_persen=(0.05, 0.25, 0.5, 0.75, 0.95)):
    roi_df = load_roi_done()
    row = roi_df[roi_df["Rekaman"] == nama_rekaman].iloc[0]
    path_video = find_video_path(DATA_ROOT, nama_rekaman)

    cap = cv2.VideoCapture(str(path_video))
    total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    landmarker = make_landmarker(MODEL_PATH, min_conf=0.5)

    print(f"\n========== {nama_rekaman} ==========")
    print(f"ROI yang digambar: ({row.x1},{row.y1})-({row.x2},{row.y2}) dari frame {row.frame_w}x{row.frame_h}")

    for p in titik_persen:
        idx = int(total_frame * p)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            print(f"[gagal baca frame {idx}]")
            continue

        crop = crop_adaptif(frame, row.x1, row.y1, row.x2, row.y2, row.frame_w, row.frame_h)

        # jalankan deteksi pose PERSIS seperti saat ekstraksi, pada crop ini
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(idx * 1000 / fps)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        crop_tampil = crop.copy()
        h, w = crop_tampil.shape[:2]
        status = "TIDAK TERDETEKSI"
        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            for pt in lm:
                cv2.circle(crop_tampil, (int(pt.x * w), int(pt.y * h)), 4, (0, 255, 0), -1)
            status = "terdeteksi"

        print(f"\n--- frame {idx} ({int(p*100)}% durasi) | crop {w}x{h} | status: {status} ---")
        cv2_imshow(crop_tampil)

    landmarker.close()
    cap.release()


if RUN_DIAGNOSTIK_VISUAL:
    for nama in ["PDFE35_2", "PDFE18_1", "PDFE07_3"]:
        diagnosa_visual_roi(nama)
else:
    print("RUN_DIAGNOSTIK_VISUAL=False -> dilewati.")

**Konteks:** Memeriksa jumlah file `.npz` dan mengelompokkan rekaman menurut fraksi frame valid (<0,25, 0,25–0,50, 0,50–0,80, ≥0,80). Hasilnya disimpan sebagai `metadata_kualitas_pose.csv`.

In [ ]:
# Cek jumlah .npz dan kelompokkan rekaman menurut kualitas pose
import numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# path dari Sel 0.3

n_npz = len(list(KEYPOINT_DIR.glob("*.npz")))
print(f"File keypoint (.npz): {n_npz} / 73")

meta = pd.read_csv(OUT_CSV_ROI)
meta = meta[meta["Rekaman"] != "Median seluruh rekaman"].copy()
meta["fraksi_valid"] = meta["Frame dengan kedua pergelangan kaki terdeteksi (%)"] / 100

bins = [-0.001, 0.25, 0.50, 0.80, 1.0001]
labels = ["<0,25", "0,25-0,50", "0,50-0,80", ">=0,80"]
meta["kelompok_kualitas"] = pd.cut(meta["fraksi_valid"], bins=bins, labels=labels, right=False)
meta["subjek"] = meta["Rekaman"].str.extract(r"PDFE(\d+)_")[0].astype(int)

print(meta["kelompok_kualitas"].value_counts().reindex(labels))
meta[["Rekaman", "subjek", "fraksi_valid", "kelompok_kualitas"]].to_csv(
    BASE / "metadata_kualitas_pose.csv", index=False)

**Konteks:** Fungsi memetakan nama rekaman (mis. `PDFE18_1`) ke file IMU pasangannya `SUB18_1.txt`.

In [ ]:
# Cari file IMU pasangan tiap rekaman
import re

# IMU_ROOT dari Sel 0.3
print("File IMU ditemukan:", len(list(IMU_ROOT.rglob("SUB*_*.txt"))))

def nama_ke_imu_path(nama_rekaman: str) -> Path:
    """'PDFE18_1' -> SUB18_1.txt di IMU_ROOT."""
    m = re.match(r"PDFE(\d+)_(\d+)", nama_rekaman)
    if not m:
        raise ValueError(f"Nama rekaman tidak dikenali: {nama_rekaman}")
    subj, sesi = int(m.group(1)), int(m.group(2))
    kandidat = list(IMU_ROOT.rglob(f"SUB{subj:02d}_{sesi}.txt"))
    if not kandidat:
        raise FileNotFoundError(f"IMU untuk {nama_rekaman} (SUB{subj:02d}_{sesi}.txt) tidak ditemukan")
    return kandidat[0]

**Konteks:** Memeriksa bahwa setiap rekaman pose punya pasangan file IMU dan mencatat rekaman yang tidak punya.

In [ ]:
# Cell A – hitung hanya file trial, lalu cek 73 rekaman punya pasangan IMU
trial_files = [p for p in IMU_ROOT.rglob("SUB*.txt") if re.match(r"^SUB\d{2}_\d\.txt$", p.name)]
print("File IMU trial:", len(trial_files), "| standing:", len(list(IMU_ROOT.rglob("SUB*_standing.txt"))))

nama_semua = sorted(p.stem for p in Path(KEYPOINT_DIR).glob("PDFE*.npz"))   # 73 nama rekaman dari npz
tanpa_imu = []
for n in nama_semua:
    try:
        nama_ke_imu_path(n)
    except Exception as e:
        tanpa_imu.append(n)
print("Rekaman tanpa pasangan IMU:", len(tanpa_imu), tanpa_imu)

**Konteks:** Loader IMU (waktu, akselerometer, giroskop, label FoG) dan penetapan `REKAMAN_VALID`, yaitu rekaman yang punya IMU. Dilengkapi satu contoh pemanggilan.

In [ ]:
# Cell D – loader IMU memakai kolom Time asli, dan daftar rekaman valid (punya IMU)
def muat_imu(nama_rekaman):
    """t_ms, acc (n,3), gyro (n,3), label (n,) dari file IMU."""
    p = nama_ke_imu_path(nama_rekaman)
    df = pd.read_csv(p, sep="\t", header=0)
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    t_ms  = df.iloc[:, 1].to_numpy() * 1000.0
    acc   = df.iloc[:, 2:5].to_numpy()
    gyro  = df.iloc[:, 5:8].to_numpy()
    label = (df.iloc[:, 8].to_numpy() > 0).astype(int)
    return t_ms, acc, gyro, label

REKAMAN_VALID = [n for n in nama_semua if n not in tanpa_imu]
print("Rekaman dipakai untuk deteksi:", len(REKAMAN_VALID))

contoh = "PDFE18_1" if "PDFE18_1" in REKAMAN_VALID else REKAMAN_VALID[0]
t, a, g, y = muat_imu(contoh)
print(contoh, t[:3], a.shape, g.shape, "| FoG:", y.sum(), "dari", len(y), "sampel")

**Konteks:** Resampling pose, IMU, dan label ke grid bersama 32 Hz (interpolasi linear, label nearest), lalu menyimpan satu parquet per rekaman.

In [ ]:
# Sel 3 – resampling pose + IMU ke grid 32 Hz bersama, simpan parquet per rekaman
from scipy.interpolate import interp1d

# RESAMPLED_DIR dari Sel 0.3
FS_TARGET = 32
STEP_MS = 1000 / FS_TARGET          # 31.25 ms

def resample_rekaman(nama):
    d = np.load(Path(KEYPOINT_DIR) / f"{nama}.npz")
    t_pose = d["t_ms"]
    xy  = d["xy"].reshape(len(t_pose), -1)      # (n,66)
    vis = d["vis"]                              # (n,33)
    t_imu, acc, gyro, label = muat_imu(nama)

    t_mulai = max(t_pose[0], t_imu[0])
    t_akhir = min(t_pose[-1], t_imu[-1])
    t_grid  = np.arange(t_mulai, t_akhir, STEP_MS)

    lin = lambda t, v: interp1d(t, v, axis=0, kind="linear", bounds_error=False)(t_grid)
    xy_r, vis_r = lin(t_pose, xy), lin(t_pose, vis)
    acc_r, gyro_r = lin(t_imu, acc), lin(t_imu, gyro)
    y_r = interp1d(t_imu, label, kind="nearest", bounds_error=False,
                   fill_value=(label[0], label[-1]))(t_grid).astype(int)

    df = pd.DataFrame(xy_r, columns=[f"xy_{i}" for i in range(66)])
    for i in range(33): df[f"vis_{i}"] = vis_r[:, i]
    for j, c in enumerate(["acc_ml","acc_ap","acc_si"]):  df[c] = acc_r[:, j]
    for j, c in enumerate(["gyr_ml","gyr_ap","gyr_si"]):  df[c] = gyro_r[:, j]
    df["t_ms"], df["label"] = t_grid, y_r
    df["valid_ankles"] = (~np.isnan(xy_r[:, 54]) & ~np.isnan(xy_r[:, 56])).astype(int)
    df.to_parquet(RESAMPLED_DIR / f"{nama}.parquet")
    return len(df), y_r.mean(), df["valid_ankles"].mean()

ringkas = []
for n in tqdm(REKAMAN_VALID):
    if (RESAMPLED_DIR / f"{n}.parquet").exists():
        continue
    ringkas.append((n, *resample_rekaman(n)))
print("Baru diproses:", len(ringkas), "| total parquet:", len(list(RESAMPLED_DIR.glob("*.parquet"))))
print(pd.DataFrame(ringkas, columns=["rekaman","n_sampel","prop_FoG","frac_valid_ankles"]).describe().round(3))

**Konteks:** Diagnostik label per rekaman dan per subjek: prevalensi FoG, rekaman tanpa FoG, dan subjek yang punya kedua kelas. Hanya membaca kolom ringan.

In [ ]:
# Cell 3b – diagnostik label FoG per rekaman dan per subjek (hanya baca kolom ringan)
rows = []
for n in REKAMAN_VALID:
    df = pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet", columns=["label", "valid_ankles", "t_ms"])
    rows.append((n, int(re.match(r"PDFE(\d+)_", n).group(1)), len(df),
                 df.label.mean(), df.valid_ankles.mean(), df.t_ms.iloc[-1] / 1000))
tbl = pd.DataFrame(rows, columns=["rekaman", "subjek", "n", "prop_FoG", "frac_valid", "durasi_s"])

print("Prevalensi FoG gabungan:", round(np.average(tbl.prop_FoG, weights=tbl.n), 3))
print("Rekaman tanpa FoG:", (tbl.prop_FoG == 0).sum(), "| FoG >= 90%:", (tbl.prop_FoG >= 0.9).sum())
print(tbl[tbl.prop_FoG >= 0.9][["rekaman", "prop_FoG", "frac_valid"]])
print(tbl.sort_values("durasi_s").head(3)[["rekaman", "durasi_s"]])

tbl["n_fog"] = tbl.prop_FoG * tbl.n
per_subj = tbl.groupby("subjek")[["n_fog", "n"]].sum()
print("Subjek dengan FoG:", (per_subj.n_fog > 0).sum(),
      "| punya kedua kelas:", ((per_subj.n_fog > 0) & (per_subj.n_fog < per_subj.n)).sum(),
      "dari", len(per_subj))

**Konteks:** Smoke test: fitur kinematik sederhana (kecepatan sendi bawah, jendela 1 s) dan satu fold LOSO dengan regresi logistik. Bukan replikasi Li dkk. dan tidak ada kebocoran: normalisasi dari pasien latih.

In [ ]:
# Sel 4 – fitur kinematik sederhana + satu fold LOSO (smoke test, bukan replikasi Li dkk.)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

KP_BAWAH = [23, 24, 25, 26, 27, 28]    # pinggul, lutut, pergelangan kaki (kiri, kanan)

def hitung_fitur(df):
    """Rerata |kecepatan| dan SD kecepatan, jendela 1 s."""
    xy = df[[f"xy_{i}" for i in range(66)]].to_numpy().reshape(len(df), 33, 2)
    low = xy[:, KP_BAWAH, :].reshape(len(df), -1)                       # (n,12)
    v = np.vstack([np.full((1, 12), np.nan), np.diff(low, axis=0)]) * FS_TARGET
    f_mean = pd.DataFrame(np.abs(v)).rolling(32, min_periods=8, center=True).mean().to_numpy()
    f_std  = pd.DataFrame(v).rolling(32, min_periods=8, center=True).std().to_numpy()
    return np.hstack([f_mean, f_std])                                   # (n,24)

def bangun_dataset(daftar):
    X, y, subj = [], [], []
    for n in daftar:
        df = pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet")
        X.append(hitung_fitur(df)); y.append(df["label"].to_numpy())
        subj += [int(re.match(r"PDFE(\d+)_", n).group(1))] * len(df)
    return np.vstack(X), np.concatenate(y), np.array(subj)

X_all, y_all, s_all = bangun_dataset(REKAMAN_VALID)
print("Dataset:", X_all.shape, "| prop FoG:", y_all.mean().round(3), "| subjek:", len(np.unique(s_all)))

def jalankan_fold(subjek_uji, seed=0):
    """Satu fold LOSO: uji pada subjek_uji, latih pada sisanya."""
    te = s_all == subjek_uji
    tr = ~te
    assert not (set(s_all[tr]) & set(s_all[te])), "LEAK: subjek uji ada di data latih"
    mu, sd = np.nanmean(X_all[tr], 0), np.nanstd(X_all[tr], 0) + 1e-8   # hanya dari latih
    prep = lambda X: np.nan_to_num((X - mu) / sd, nan=0.0)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed)
    clf.fit(prep(X_all[tr]), y_all[tr])
    p = clf.predict_proba(prep(X_all[te]))[:, 1]
    return dict(subjek_uji=subjek_uji, n_uji=int(te.sum()), prop_FoG=y_all[te].mean(),
                auroc=roc_auc_score(y_all[te], p), auprc=average_precision_score(y_all[te], p))

kandidat = [s for s in np.unique(s_all) if 0 < y_all[s_all == s].sum() < (s_all == s).sum()]
print("Subjek dengan kedua kelas:", len(kandidat), "dari", len(np.unique(s_all)))

prev = {s: y_all[s_all == s].mean() for s in kandidat}
uji = [s for s in kandidat if 0.1 <= prev[s] <= 0.6]
print("Kandidat smoke test (prevalensi 10-60%):", uji[:10])
print(jalankan_fold(uji[0]))

**Konteks:** Menjalankan semua fold LOSO (1 seed) dan membandingkan AUROC per pasien dengan kualitas pose subjek, termasuk korelasi Spearman.

In [ ]:
# Cell 5 – jalankan semua fold LOSO (1 seed) dan bandingkan AUROC dengan kualitas pose subjek
hasil = []
for s in kandidat:
    r = jalankan_fold(s)
    r["frac_valid"] = np.average(tbl[tbl.subjek == s].frac_valid, weights=tbl[tbl.subjek == s].n)
    hasil.append(r)
hasil = pd.DataFrame(hasil).sort_values("auroc")
hasil["auprc_vs_acak"] = hasil.auprc - hasil.prop_FoG
print(hasil[["subjek_uji", "prop_FoG", "frac_valid", "auroc", "auprc_vs_acak"]].round(3).to_string(index=False))
print("\nAUROC per pasien: rerata %.3f | SD %.3f | median %.3f" %
      (hasil.auroc.mean(), hasil.auroc.std(), hasil.auroc.median()))
print("Fold dengan AUROC > 0.6:", (hasil.auroc > 0.6).sum(), "dari", len(hasil))
print("Spearman AUROC vs frac_valid:", round(hasil[["auroc", "frac_valid"]].corr(method="spearman").iloc[0, 1], 3))

**Konteks:** Mengestimasi selisih waktu video vs IMU lewat korelasi silang antara kecepatan pergelangan kaki dan giroskop.

In [ ]:
# Cell 6 – estimasi selisih waktu video vs IMU lewat korelasi silang envelope aktivitas
def lag_rekaman(df, maks=96):
    """Lag (sampel 32 Hz) dengan korelasi tertinggi kecepatan ankle vs gyro."""
    a = df[["xy_54", "xy_55", "xy_56", "xy_57"]].to_numpy()          # x,y ankle kiri & kanan
    dL = np.hypot(np.diff(a[:, 0]), np.diff(a[:, 1]))
    dR = np.hypot(np.diff(a[:, 2]), np.diff(a[:, 3]))
    with np.errstate(all="ignore"):
        spd = np.nanmean(np.vstack([dL, dR]), axis=0)
    gyr = np.sqrt((df[["gyr_ml", "gyr_ap", "gyr_si"]].to_numpy() ** 2).sum(1))[1:]
    env = lambda x: pd.Series(x).rolling(32, min_periods=8, center=True).mean().to_numpy()
    zs = lambda x: np.nan_to_num((x - np.nanmean(x)) / (np.nanstd(x) + 1e-8), nan=0.0)
    p, g = zs(env(spd)), zs(env(gyr))
    kor = {}
    for k in range(-maks, maks + 1):
        x, y = (p[k:], g[:len(g) - k]) if k >= 0 else (p[:k], g[-k:])
        kor[k] = np.mean(x * y)
    k_best = max(kor, key=kor.get)
    return k_best / FS_TARGET, kor[k_best], kor[0]

rows = []
for n in tbl[tbl.frac_valid >= 0.7].rekaman:
    df = pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet")
    lag_s, kmax, k0 = lag_rekaman(df)
    rows.append((n, round(lag_s, 2), round(kmax, 2), round(k0, 2)))
lagdf = pd.DataFrame(rows, columns=["rekaman", "lag_detik", "korelasi_maks", "korelasi_lag0"])
print(lagdf.to_string(index=False))
print("\nMedian lag: %.2f s | IQR: %.2f s sampai %.2f s | n=%d" %
      (lagdf.lag_detik.median(), lagdf.lag_detik.quantile(.25), lagdf.lag_detik.quantile(.75), len(lagdf)))

### Sel 4.4 — Simpan hasil Sel 4.2 dan 4.3 ke Drive
**Apa:** menulis `hasil_loso_kinematik_seed0.csv`, `hasil_lag_sinkronisasi.csv`, `ringkasan_rekaman_71.csv` ke `05. Week 2`. (Sel 4.5 membaca file pertama.)

In [ ]:
# Cell 7 – simpan hasil Cell 5 dan Cell 6 ke Drive supaya tidak hilang saat Colab terputus
hasil.to_csv(Path(BASE) / "hasil_loso_kinematik_seed0.csv", index=False)
lagdf.to_csv(Path(BASE) / "hasil_lag_sinkronisasi.csv", index=False)
tbl.drop(columns=["n_fog"], errors="ignore").to_csv(Path(BASE) / "ringkasan_rekaman_71.csv", index=False)
print("Tersimpan di:", BASE)
print([p.name for p in Path(BASE).glob("hasil_*.csv")] + ["ringkasan_rekaman_71.csv"])

### Sel 4.5 — Sensitivitas pergeseran waktu +0,44 s
**Apa:** apakah koreksi offset memperbaiki AUROC? Pergeseran ditetapkan **di muka** dari Sel 4.3 (bukan dipilih dari AUROC uji).
**Hasil yang diharapkan:** AUROC 0,587 vs baseline 0,597; selisih −0,010; membaik/memburuk 12/14; Wilcoxon p = 0,217 → **tidak ada perbaikan**; penyelarasan bukan penyebab utama sinyal lemah.

**Konteks:** Uji sensitivitas: menggeser fitur pose terhadap IMU sebesar lag terestimasi lalu mengulang LOSO dan membandingkan dengan baseline memakai Wilcoxon.

In [ ]:
# Cell 8 – sensitivitas pergeseran waktu pose-IMU (LOSO 26 subjek, seed 0)
from scipy.stats import wilcoxon

hasil = pd.read_csv(Path(BASE) / "hasil_loso_kinematik_seed0.csv")   # baseline (shift 0) dari Cell 5

def geser_fitur(F, k):
    """Geser fitur k sampel; ujung diisi NaN."""
    if k == 0:
        return F
    out = np.full_like(F, np.nan)
    if k > 0: out[:-k] = F[k:]
    else:     out[-k:] = F[:k]
    return out

def bangun_dataset_geser(daftar, k):
    X, y, subj = [], [], []
    for n in daftar:
        df = pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet")
        X.append(geser_fitur(hitung_fitur(df), k)); y.append(df["label"].to_numpy())
        subj += [int(re.match(r"PDFE(\d+)_", n).group(1))] * len(df)
    return np.vstack(X), np.concatenate(y), np.array(subj)

SHIFTS_DETIK = [0.44]          # ditetapkan di muka dari Cell 6; tambah -0.44 dan 1.0 bila waktu cukup
base = hasil.set_index("subjek_uji")["auroc"]
ringkas_shift = []
for sh in SHIFTS_DETIK:
    k = int(round(sh * FS_TARGET))
    X_all, y_all, s_all = bangun_dataset_geser(REKAMAN_VALID, k)
    hs = pd.DataFrame([jalankan_fold(s) for s in kandidat]).set_index("subjek_uji")
    hs["auprc_vs_acak"] = hs.auprc - hs.prop_FoG
    hs[["auroc", "auprc_vs_acak"]].to_csv(Path(BASE) / f"hasil_loso_shift_{sh:+.2f}s.csv")
    d = hs.auroc - base.loc[hs.index]
    ringkas_shift.append(dict(shift_detik=sh, k_sampel=k, auroc_rerata=hs.auroc.mean(), auroc_sd=hs.auroc.std(),
                              selisih_rerata=d.mean(), n_membaik=int((d > 0).sum()),
                              n_memburuk=int((d < 0).sum()), p_wilcoxon=wilcoxon(d).pvalue))

X_all, y_all, s_all = bangun_dataset(REKAMAN_VALID)      # kembalikan dataset global ke shift 0
print("Baseline shift 0: AUROC rerata %.3f | SD %.3f" % (base.mean(), base.std()))
print(pd.DataFrame(ringkas_shift).round(3).to_string(index=False))

**Konteks:** Baseline hanya IMU dengan classifier dan fold yang sama seperti Sel 4.2, memakai 12 fitur (rerata |x| dan SD, jendela 1 s) dari 6 kanal IMU.

In [ ]:
# Cell 9 – baseline IMU-only: classifier dan fold sama dengan Cell 5, fitur dari IMU (LOSO 26 subjek, seed 0)
IMU_KOL = ["acc_ml", "acc_ap", "acc_si", "gyr_ml", "gyr_ap", "gyr_si"]

def hitung_fitur_imu(df):
    """Rerata |x| dan SD x, jendela 1 s, 6 kanal -> 12 fitur."""
    imu = df[IMU_KOL].to_numpy()
    f_mean = pd.DataFrame(np.abs(imu)).rolling(32, min_periods=8, center=True).mean().to_numpy()
    f_std  = pd.DataFrame(imu).rolling(32, min_periods=8, center=True).std().to_numpy()
    return np.hstack([f_mean, f_std])

X_all, y_all, s_all = [], [], []
for n in REKAMAN_VALID:
    df = pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet")
    X_all.append(hitung_fitur_imu(df)); y_all.append(df["label"].to_numpy())
    s_all += [int(re.match(r"PDFE(\d+)_", n).group(1))] * len(df)
X_all, y_all, s_all = np.vstack(X_all), np.concatenate(y_all), np.array(s_all)
print("Dataset IMU:", X_all.shape)

hi = pd.DataFrame([jalankan_fold(s) for s in kandidat]).set_index("subjek_uji")
hi["auprc_vs_acak"] = hi.auprc - hi.prop_FoG
hi[["auroc", "auprc_vs_acak"]].to_csv(Path(BASE) / "hasil_loso_imu_only_seed0.csv")

base = hasil.set_index("subjek_uji")["auroc"]
d = hi.auroc - base.loc[hi.index]
print("IMU-only  : AUROC rerata %.3f | SD %.3f | median %.3f" % (hi.auroc.mean(), hi.auroc.std(), hi.auroc.median()))
print("Pose-only : AUROC rerata %.3f | SD %.3f" % (base.mean(), base.std()))
print("Selisih (IMU - pose): rerata %.3f | IMU lebih baik di %d dari %d pasien | Wilcoxon p = %.4f" %
      (d.mean(), (d > 0).sum(), len(d), wilcoxon(d).pvalue))
print("AUPRC di atas baseline acak (IMU): rerata %.3f | fold > +0.05: %d dari %d" %
      (hi.auprc_vs_acak.mean(), (hi.auprc_vs_acak > 0.05).sum(), len(hi)))

X_all, y_all, s_all = bangun_dataset(REKAMAN_VALID)      # kembalikan dataset global ke fitur pose (shift 0)

**Konteks:** Fusi cepat: konkatenasi awal (fitur pose + IMU) dan rerata akhir (rata-rata probabilitas), dibandingkan lewat LOSO. Normalisasi dihitung dari pasien latih saja dan ada assert anti-kebocoran.

In [ ]:
# Cell 10 – fusi cepat: konkatenasi awal vs rerata akhir (LOSO 26 subjek, seed 0)
from scipy.stats import spearmanr

X_pose, y_all, s_all = bangun_dataset(REKAMAN_VALID)
X_imu = np.vstack([hitung_fitur_imu(pd.read_parquet(RESAMPLED_DIR / f"{n}.parquet")) for n in REKAMAN_VALID])
X_cat = np.hstack([X_pose, X_imu])                       # konkatenasi awal: 24 + 12 = 36 fitur

def prob_fold(X, subjek_uji, seed=0):
    """Probabilitas FoG pasien uji; latih pada pasien lain."""
    te = s_all == subjek_uji
    tr = ~te
    assert not (set(s_all[tr]) & set(s_all[te])), "LEAK"
    mu, sd = np.nanmean(X[tr], 0), np.nanstd(X[tr], 0) + 1e-8
    prep = lambda A: np.nan_to_num((A - mu) / sd, nan=0.0)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed).fit(prep(X[tr]), y_all[tr])
    return clf.predict_proba(prep(X[te]))[:, 1]

rows = []
for s in kandidat:
    y = y_all[s_all == s]
    p_pose, p_imu, p_cat = prob_fold(X_pose, s), prob_fold(X_imu, s), prob_fold(X_cat, s)
    p_late = (p_pose + p_imu) / 2                        # rerata akhir
    rows.append(dict(subjek=s, prop_FoG=y.mean(),
                     pose=roc_auc_score(y, p_pose), imu=roc_auc_score(y, p_imu),
                     early=roc_auc_score(y, p_cat), late=roc_auc_score(y, p_late)))
fus = pd.DataFrame(rows).set_index("subjek")
fus.to_csv(Path(BASE) / "hasil_fusi_cepat_seed0.csv")

print(fus[["pose", "imu", "early", "late"]].agg(["mean", "std"]).round(3))
for nama in ("early", "late"):
    d = fus[nama] - fus["imu"]
    print(f"{nama} - IMU: rerata {d.mean():+.3f} | membaik {int((d>0).sum())}/{len(d)} | Wilcoxon p = {wilcoxon(d).pvalue:.4f}")
fv = hasil.set_index("subjek_uji")["frac_valid"].loc[fus.index]
print("Spearman (early - IMU) vs frac_valid: %.3f" % spearmanr(fus.early - fus.imu, fv)[0])

**Konteks:** Menghitung fitur kinematik gaya Li dkk. (2022) pada grid 32 Hz (jendela 2 s, statistik per sendi dan pasangan kiri-kanan), disimpan per rekaman ke `fitur_li_v2`.

In [ ]:
# Cell 11 – fitur kinematik gaya Li dkk. (2022) pada grid evaluasi 32 Hz, disimpan per rekaman ke Drive
import json, warnings
from numpy.lib.stride_tricks import sliding_window_view
from scipy.interpolate import interp1d

FS, WIN = 32, 64                                   # 32 Hz; jendela 2 detik = 64 sampel
KP_IDX  = [23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
KP_NAMA = ["pinggulL","pinggulR","lututL","lututR","pergelanganL","pergelanganR","tumitL","tumitR","jempolL","jempolR"]
PASANGAN = [(0,1,"pinggul"), (2,3,"lutut"), (4,5,"pergelangan"), (6,7,"tumit"), (8,9,"jempol")]
STAT = ["mean","std","min","max","ptp","skew","kurt","crest","pita_rendah","pita_tinggi","FI","amp_puncak","frek_dominan"]

def dimensi_crop(x1, y1, x2, y2, frame_w, frame_h, frac=0.15):
    """Ukuran crop (piksel), sama dengan crop_adaptif."""
    pad_x, pad_y = int((x2 - x1) * frac), int((y2 - y1) * frac)
    x1p, y1p = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    x2p, y2p = min(int(frame_w), int(x2 + pad_x)), min(int(frame_h), int(y2 + pad_y))
    return x2p - x1p, y2p - y1p

def _sudut(a, b, c):
    """Sudut di b (derajat) antara b->a dan b->c."""
    v1, v2 = a - b, c - b
    n = np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)
    with np.errstate(all="ignore"):
        cos = np.clip((v1 * v2).sum(1) / n, -1, 1)
    return np.degrees(np.arccos(cos))

def bangun_sinyal(xy):
    """xy (K,33,2) -> sinyal (K,S) + nama sinyal."""
    P = xy[:, KP_IDX, :]
    V = np.full_like(P, np.nan); V[1:] = (P[1:] - P[:-1]) * FS
    A = np.full_like(P, np.nan); A[1:] = (V[1:] - V[:-1]) * FS
    with np.errstate(all="ignore"):
        spd, acc = np.linalg.norm(V, axis=2), np.linalg.norm(A, axis=2)
    sig, nama = [], []
    for j, n in enumerate(KP_NAMA): sig.append(spd[:, j]); nama.append(f"kecepatan_{n}")
    for j, n in enumerate(KP_NAMA): sig.append(acc[:, j]); nama.append(f"percepatan_{n}")
    sig.append(_sudut(xy[:, 23], xy[:, 25], xy[:, 27])); nama.append("sudut_lutut_L")
    sig.append(_sudut(xy[:, 24], xy[:, 26], xy[:, 28])); nama.append("sudut_lutut_R")
    for a, b, n in PASANGAN:
        for ax, an in ((0, "x"), (1, "y")):
            sig.append(np.abs(P[:, a, ax] - P[:, b, ax])); nama.append(f"selisihLR_{n}_{an}")
    return np.stack(sig, axis=1), nama

def fitur_jendela(sig):
    """Sinyal (K,S) -> fitur waktu-frekuensi (K,S*13), jendela 2 s."""
    K, S = sig.shape; mp = WIN // 2
    df = pd.DataFrame(sig); r = df.rolling(WIN, center=True, min_periods=mp)
    mean, std, mn, mx = r.mean().to_numpy(), r.std().to_numpy(), r.min().to_numpy(), r.max().to_numpy()
    skew, kurt = r.skew().to_numpy(), r.kurt().to_numpy()
    rms = np.sqrt((df ** 2).rolling(WIN, center=True, min_periods=mp).mean().to_numpy())
    crest = df.abs().rolling(WIN, center=True, min_periods=mp).max().to_numpy() / (rms + 1e-9)
    freqs = np.fft.rfftfreq(WIN, 1 / FS)
    b_lo, b_hi, b_fz = (freqs >= 0.5) & (freqs <= 3.0), (freqs >= 3.5) & (freqs <= 15.0), (freqs >= 3.0) & (freqs <= 8.0)
    frek = np.full((K, S, 5), np.nan)
    pad = np.vstack([np.full((WIN // 2, S), np.nan), sig, np.full((WIN // 2 - 1, S), np.nan)])
    for s in range(S):
        w = sliding_window_view(pad[:, s], WIN)
        ok = (~np.isnan(w)).mean(1) >= 0.5
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); mu = np.nanmean(w, axis=1)
        w = np.where(np.isnan(w), mu[:, None], w) - mu[:, None]
        with np.errstate(all="ignore"):
            Fk = np.fft.rfft(w, axis=1); Pw = (np.abs(Fk) ** 2) / WIN
            lo, hi, fz = Pw[:, b_lo].sum(1), Pw[:, b_hi].sum(1), Pw[:, b_fz].sum(1)
            fi = np.clip(fz / (lo + 1e-9), 0, 1e6)
            amp = np.abs(Fk[:, 1:]).max(1); domf = freqs[1:][np.abs(Fk[:, 1:]).argmax(1)]
        blok = np.stack([lo, hi, fi, amp, domf], axis=1); blok[~ok] = np.nan
        frek[:, s, :] = blok
    waktu = np.stack([mean, std, mn, mx, mx - mn, skew, kurt, crest], axis=2)
    return np.concatenate([waktu, frek], axis=2).reshape(K, S * len(STAT)).astype(np.float32)

def fitur_rekaman(npz_path, t_grid_ms, crop_w, crop_h):
    d = np.load(npz_path)
    xy = d["xy"].reshape(len(d["t_ms"]), -1)
    g = interp1d(d["t_ms"], xy, axis=0, kind="linear", bounds_error=False)(t_grid_ms).reshape(len(t_grid_ms), 33, 2).copy()
    g[:, :, 0] *= crop_w / crop_h                         # satuan = tinggi crop (isotropik)
    sig, nama_sig = bangun_sinyal(g)
    return fitur_jendela(sig), [f"{n}__{s}" for n in nama_sig for s in STAT]

# jalankan untuk 71 rekaman (resumable)
# FITUR_DIR dari Sel 0.3
roi = pd.read_csv(ROI_CSV); roi["kunci"] = roi["Rekaman"].astype(str).map(lambda s: Path(s).stem); roi = roi.set_index("kunci")
nan_frac = {}
for n in tqdm(REKAMAN_VALID):
    out = FITUR_DIR / f"{n}.npz"
    if out.exists(): continue
    t_ms, _, _, label = muat_imu(n)
    K = len(label) // 4                                    # sama dengan panjang label pada 32 Hz di fogbench
    cw, ch = dimensi_crop(*roi.loc[n, ["x1","y1","x2","y2","frame_w","frame_h"]])
    X, nama_fitur = fitur_rekaman(Path(KEYPOINT_DIR) / f"{n}.npz", t_ms[::4][:K], cw, ch)
    np.savez_compressed(out, X=X); nan_frac[n] = float(np.isnan(X).mean())
    if not (FITUR_DIR / "nama_fitur.json").exists():
        (FITUR_DIR / "nama_fitur.json").write_text(json.dumps(nama_fitur))
print("Berkas fitur:", len(list(FITUR_DIR.glob("PDFE*.npz"))), "| fitur per sampel:", len(nama_fitur) if nan_frac else "(sudah ada)")
print("Fraksi NaN per rekaman (baru dihitung):", pd.Series(nan_frac).describe().round(3).to_dict() if nan_frac else "-")

**Konteks:** Memasang repo `fog-benchamark-starter` (fogbench), mengatur `paths.local.yaml`, lalu memuat 71 rekaman yang punya fitur sebagai `recs`.

In [ ]:
# Pasang fogbench dan muat 71 rekaman
import os, sys, subprocess
REKAMAN_VALID = sorted(p.stem for p in FITUR_DIR.glob("PDFE*.npz"))     # 71 rekaman yang punya fitur
REPO = "/content/fog-benchamark-starter"                                # (ejaan "benchamark" mengikuti nama repo asli)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bungagana/fog-benchamark-starter", REPO], check=True)
Path(REPO, "config", "paths.local.yaml").write_text(f'figshare: "{FIGSHARE_DIR}"\n', encoding="utf-8")
if REPO not in sys.path: sys.path.insert(0, REPO)
from fogbench import data as D

recs = D.load("figshare")
nama_recs = {f"PDFE{r.subject:02d}_{r.session + 1}" for r in recs}
print("Rekaman:", len(recs), "| pasien:", len({r.subject for r in recs}))
print("Beda dengan REKAMAN_VALID:", sorted(nama_recs ^ set(REKAMAN_VALID)))
salah = []
for r in recs:
    n = f"PDFE{r.subject:02d}_{r.session + 1}"
    K = len(D.to_eval_rate(r.y, r.fs, 32, is_label=True))
    Kf = len(np.load(FITUR_DIR / f"{n}.npz")["X"])
    if K != Kf: salah.append((n, Kf, K))
print("Panjang fitur != panjang label:", salah)

**Konteks:** Menulis `models/li_kinematik_xgb.py`: model Li (fitur kinematik + seleksi fitur top-K + XGBoost) sesuai antarmuka `FoGModel` fogbench.

In [ ]:
%%writefile /content/fog-benchamark-starter/models/li_kinematik_xgb.py
"""Fitur kinematik pose + XGBoost (Li dkk., 2022)."""
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from fogbench import data as D
from models.base import FoGModel

FS = 32
FEATURE_DIR = Path(os.environ.get("FOG_FITUR_DIR", "fitur_li_v2"))
STRIDE_LATIH = 3          # 3 sampel pada 32 Hz = ~0,1 detik, sama dengan langkah jendela Li dkk.
TOP_K = 20
SMOOTH_DETIK = 1.0        # penghalusan probabilitas per rekaman (rerata bergerak terpusat)
_CACHE: dict = {}


def _muat(subject: int, session: int) -> np.ndarray:
    key = (subject, session)
    if key not in _CACHE:
        _CACHE[key] = np.load(FEATURE_DIR / f"PDFE{subject:02d}_{session + 1}.npz")["X"].astype(np.float32)
    return _CACHE[key]


def _selaraskan(X: np.ndarray, n: int) -> np.ndarray:
    """Potong atau isi NaN sampai panjang n."""
    if len(X) >= n:
        return X[:n]
    return np.vstack([X, np.full((n - len(X), X.shape[1]), np.nan, np.float32)])


class LiKinematikXGB(FoGModel):
    name = "li-kinematik-xgb"

    def __init__(self, seed: int = 0, n_pohon: int = 200, kedalaman: int = 4):
        self.seed, self.n_pohon, self.kedalaman = seed, n_pohon, kedalaman
        self.idx = None
        self.model = None

    def _xgb(self, n_pohon, spw):
        return XGBClassifier(n_estimators=n_pohon, max_depth=self.kedalaman, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                             scale_pos_weight=spw, tree_method="hist", n_jobs=-1,
                             random_state=self.seed, eval_metric="logloss", verbosity=0)

    def fit(self, recordings) -> None:
        Xs, ys = [], []
        for r in recordings:
            y32 = D.to_eval_rate(r.y, r.fs, FS, is_label=True)
            X = _selaraskan(_muat(r.subject, r.session), len(y32))
            Xs.append(X[::STRIDE_LATIH]); ys.append(np.asarray(y32)[::STRIDE_LATIH])
        X, y = np.vstack(Xs), np.concatenate(ys).astype(int)
        n_pos = max(int(y.sum()), 1)
        spw = (len(y) - n_pos) / n_pos
        # seleksi fitur HANYA dari pasien latih: peringkat gain XGBoost, ambil TOP_K
        pilih = self._xgb(100, spw).fit(X, y)
        gain = pilih.get_booster().get_score(importance_type="gain")
        urut = sorted(gain, key=gain.get, reverse=True)[:TOP_K]
        self.idx = np.array(sorted(int(k[1:]) for k in urut))
        self.model = self._xgb(self.n_pohon, spw).fit(X[:, self.idx], y)

    def predict_proba(self, recording) -> np.ndarray:
        n128 = len(recording.y)
        n32 = len(D.to_eval_rate(recording.y, recording.fs, FS, is_label=True))
        X = _selaraskan(_muat(recording.subject, recording.session), n32)[:, self.idx]
        p32 = self.model.predict_proba(X)[:, 1]
        if SMOOTH_DETIK > 0:        # hanya memakai probabilitas rekaman ini sendiri, tanpa label
            p32 = pd.Series(p32).rolling(int(SMOOTH_DETIK * FS), center=True, min_periods=1).mean().to_numpy()
        p = np.repeat(p32, recording.fs // FS)
        if len(p) < n128:
            p = np.concatenate([p, np.full(n128 - len(p), p[-1])])
        return p[:n128].astype(np.float32)

    def card(self) -> dict:
        n_simpul = None
        if self.model is not None:
            n_simpul = int(len(self.model.get_booster().trees_to_dataframe()))
        return {
            "nama": self.name,
            "n_parameter": n_simpul,
            "n_parameter_keterangan": "jumlah simpul pohon XGBoost (model akhir, TOP_K fitur)",
            "hiperparameter": {
                "n_pohon": self.n_pohon, "kedalaman_maks": self.kedalaman, "learning_rate": 0.1,
                "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
                "scale_pos_weight": "n_negatif/n_positif dari pasien latih",
                "seleksi_fitur": f"peringkat gain XGBoost (100 pohon) dari pasien latih, ambil {TOP_K}",
                "jendela_detik": 2, "langkah_latih_sampel": STRIDE_LATIH,
                "penghalusan_probabilitas_detik": SMOOTH_DETIK, "seed": self.seed},
            "deviasi": [
                "Pose dari MediaPipe BlazePose (33 keypoint 2D), bukan OpenPose 25 keypoint seperti Li dkk. (2022).",
                "Hanya 10 keypoint tubuh bawah (pinggul, lutut, pergelangan, tumit, jempol kaki); Li dkk. memakai juga bahu, siku, pergelangan tangan, telinga, leher.",
                "Sinyal: kecepatan, percepatan, sudut lutut, selisih kiri-kanan 5 pasang (x dan y); 32 sinyal. Posisi mentah tidak dipakai sebagai sinyal.",
                "Fitur per jendela 2 detik: 8 domain waktu + 5 domain frekuensi (luas pita 0,5-3 Hz dan 3,5-15 Hz, FI = daya 3-8 Hz / daya 0,5-3 Hz, amplitudo puncak, frekuensi dominan). Daftar fitur Li dkk. lebih panjang (mis. clearance factor, mutual information).",
                "Seleksi fitur: peringkat gain XGBoost dari pasien latih (top 20), bukan forward feature selection seperti Li dkk.",
                "Tidak ada model bertahap jalan/berbalik (multi-stage); satu model FoG untuk seluruh rekaman.",
                "Li dkk. menghapus episode FoG prediksi yang lebih pendek dari kuantil 10% durasi FoG. Di sini diganti penghalusan probabilitas (rerata bergerak 1 detik per rekaman) di dalam predict_proba, karena penghapusan episode butuh ambang yang dipilih oleh penjalan eksperimen.",
                "Normalisasi koordinat: satuan = tinggi crop ROI manual per video (konstan per video, isotropik); Li dkk. memakai kotak pembatas tubuh per frame diperluas 30% dan diskalakan ke tinggi 80.",
                "Laju 32 Hz (Li dkk. 30 Hz). Fitur dihitung tiap sampel 32 Hz; pelatihan memakai langkah 3 sampel (~0,1 detik, sama dengan Li dkk.).",
                "Frame pose yang tidak terdeteksi dibiarkan NaN; XGBoost menanganinya sebagai nilai hilang. Visibility keypoint tidak dipakai untuk menyaring.",
                "Hiperparameter XGBoost tidak disebut di makalah; ditentukan sendiri tanpa penyetelan.",
                "Dataset berbeda: Figshare (berbalik di tempat, label dari IMU) vs. 50 pasien TUG di Li dkk.",
            ]}


def factory(seed: int = 0) -> LiKinematikXGB:
    return LiKinematikXGB(seed=seed)

**Konteks:** Mendefinisikan `satu_fold`, salinan logika `run_loso` fogbench untuk satu fold (pelatihan, ambang dari data latih, metrik, dan penyimpanan JSON per fold).

In [ ]:
# Fungsi satu fold LOSO dengan protokol fogbench
import os, json, time, importlib
from pathlib import Path
import numpy as np
from fogbench import data as D, splits, schema
from fogbench.threshold import pick_threshold
from fogbench.metrics import sample_metrics
from fogbench.evaluate import protocol

def satu_fold(model_factory, recs, subjek_uji, out_dir, model_name, dataset="figshare", seeds=None, fs_eval=None):
    """run_loso fogbench untuk satu fold (logika sama)."""
    p = protocol()
    seeds = list(seeds or p["seeds"]); fs_eval = int(fs_eval or p["fs_evaluasi_hz"])
    fold = next(f for f in splits.loso_folds({r.subject for r in recs}) if f.test_subject == subjek_uji)
    train, test = splits.split_recordings(recs, fold)
    p_seeds, t_train = [], 0.0
    for sd in seeds:
        model = model_factory(seed=sd)
        t0 = time.perf_counter(); model.fit(train); t_train += time.perf_counter() - t0
        p_seeds.append(np.concatenate([D.to_eval_rate(model.predict_proba(r), r.fs, fs_eval) for r in test]))
    p_mean = np.mean(p_seeds, axis=0)
    y_eval = np.concatenate([D.to_eval_rate(r.y, r.fs, fs_eval, is_label=True) for r in test])
    n = min(len(p_mean), len(y_eval)); p_mean, y_eval = p_mean[:n], y_eval[:n]; p_seeds = [q[:n] for q in p_seeds]

    model_t = model_factory(seed=seeds[0]); model_t.fit(train)          # ambang dari pasien latih saja
    y_tr = np.concatenate([D.to_eval_rate(r.y, r.fs, fs_eval, is_label=True) for r in train])
    p_tr = np.concatenate([D.to_eval_rate(model_t.predict_proba(r), r.fs, fs_eval) for r in train])
    m = min(len(y_tr), len(p_tr)); thr = pick_threshold(y_tr[:m], p_tr[:m])
    t0 = time.perf_counter(); _ = model_t.predict_proba(test[0]); infer_ms = (time.perf_counter() - t0) * 1000

    path = schema.write_fold(Path(out_dir) / f"fold_{fold.index:02d}.json", model=model_name, dataset=dataset,
                             fold=fold.index, test_subject=fold.test_subject, train_subjects=list(fold.train_subjects),
                             fs_eval=fs_eval, threshold=thr, y_true=y_eval, p_mean=p_mean, p_per_seed=p_seeds,
                             seeds=seeds, train_time_s=t_train, infer_time_ms=infer_ms, model_card=model_t.card())
    rec = json.loads(Path(path).read_text(encoding="utf-8"))
    per_seed = [sample_metrics(y_eval, q, thr) for q in p_seeds]
    return rec, sample_metrics(y_eval, p_mean, thr), per_seed, schema.validate_fold(rec), model_t
print("satu_fold siap")

**Konteks:** Uji model Li pada satu fold (subjek 4, 3 seed): memvalidasi berkas fold dan mencetak metrik, ambang, serta waktu latih/inferensi.

In [ ]:
# Uji model Li pada 1 fold (subjek 4, 3 seed)
import importlib, json, time
os.environ["FOG_FITUR_DIR"] = str(FITUR_DIR)          # model Li membaca fitur dari sini
import models.li_kinematik_xgb as M
importlib.reload(M)

SUBJEK_UJI = 4
OUT = HASIL_FOGBENCH / "li-kinematik-xgb_figshare"
rec, m_mean, m_seed, masalah, model_t = satu_fold(M.factory, recs, SUBJEK_UJI, OUT, "li-kinematik-xgb")

print("Validasi berkas fold:", masalah or "OK", "| berkas:", OUT / f"fold_{rec['fold']:02d}.json")
print("Fold %d | pasien uji %d | n sampel 32 Hz: %d | ambang: %.2f | waktu latih (3 seed): %.0f s | inferensi: %.0f ms" %
      (rec["fold"], rec["test_subject"], len(rec["y_true"]), rec["threshold"], rec["train_time_s"], rec["infer_time_ms"]))
kol = ["auroc", "auprc", "sensitivitas", "spesifisitas", "f1", "episode_f1"]
tabel = pd.DataFrame(m_seed, index=[f"seed {s}" for s in rec["seeds"]])[kol]
print(tabel.round(3).to_string())
print("\nRerata +/- SD antar seed:", {k: f"{tabel[k].mean():.3f} +/- {tabel[k].std(ddof=1):.3f}" for k in kol})
print("Pada p_mean (rerata 3 seed):", {k: round(m_mean[k], 3) for k in kol},
      "| episode benar/prediksi:", m_mean["n_episode_true"], "/", m_mean["n_episode_pred"], "| prevalensi:", round(m_mean["prevalensi"], 3))

ref_pose = pd.read_csv(Path(BASE) / "hasil_loso_kinematik_seed0.csv").set_index("subjek_uji").auroc.get(SUBJEK_UJI)
ref_imu = pd.read_csv(Path(BASE) / "hasil_loso_imu_only_seed0.csv").set_index("subjek_uji").auroc.get(SUBJEK_UJI)
print("\nPembanding AUROC subjek %d: pose sederhana (24 fitur, regresi logistik) = %.3f | IMU-only sederhana = %.3f | Li-style XGBoost (ini) = %.3f"
      % (SUBJEK_UJI, ref_pose, ref_imu, m_mean["auroc"]))

nama_fitur = json.loads((FITUR_DIR / "nama_fitur.json").read_text())
print("\n20 fitur terpilih (model seed 0):"); [print("  ", nama_fitur[i]) for i in model_t.idx]
(OUT / "kartu_metode_li-kinematik-xgb.json").write_text(json.dumps(rec["model_card"], ensure_ascii=False, indent=2), encoding="utf-8")
print("\nKartu metode: parameter (simpul pohon) =", rec["model_card"]["n_parameter"], "| deviasi:", len(rec["model_card"]["deviasi"]))

**Konteks:** Membuat cache pose 17 sendi pada grid 32 Hz (`pose_grid32`) sebagai masukan ST-GCN, termasuk dimensi crop untuk normalisasi skema piksel.

In [ ]:
# Cache pose 17 sendi pada grid 32 Hz untuk ST-GCN
import numpy as np, pandas as pd
from scipy.interpolate import interp1d
from fogbench import data as D

BP_IDX = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
# POSE_GRID_DIR dari Sel 0.3

def dimensi_crop(x1, y1, x2, y2, frame_w, frame_h, frac=0.15):
    pad_x, pad_y = int((x2 - x1) * frac), int((y2 - y1) * frac)
    x1p, y1p = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    x2p, y2p = min(int(frame_w), int(x2 + pad_x)), min(int(frame_h), int(y2 + pad_y))
    return x2p - x1p, y2p - y1p

def cache_satu(r, npz_path, crop_w, crop_h):
    n32 = len(D.to_eval_rate(r.y, r.fs, 32, is_label=True))
    t_grid = (np.arange(n32) * 4 + 1) / 128 * 1000
    d = np.load(npz_path)
    t, xy, vis = d["t_ms"], d["xy"][:, BP_IDX, :], d["vis"][:, BP_IDX]
    f = lambda a: interp1d(t, a.reshape(len(t), -1), axis=0, kind="linear", bounds_error=False)(t_grid)
    g = f(xy).reshape(n32, 17, 2)
    g[:, :, 0] *= crop_w; g[:, :, 1] *= crop_h
    v = np.nan_to_num(f(np.nan_to_num(vis))).reshape(n32, 17)
    return g.astype(np.float32), v.astype(np.float32)

roi = pd.read_csv(ROI_CSV).set_index("Rekaman")
ringkas, hilang = [], []
for r in recs:
    nama = f"PDFE{r.subject:02d}_{r.session + 1}"
    p = KEYPOINT_DIR / f"{nama}.npz"
    if nama not in roi.index or not p.exists():
        hilang.append(nama); continue
    row = roi.loc[nama]
    cw, ch = dimensi_crop(row.x1, row.y1, row.x2, row.y2, row.frame_w, row.frame_h)
    g, v = cache_satu(r, p, cw, ch)
    np.savez_compressed(POSE_GRID_DIR / f"{nama}.npz", xy=g, vis=v)
    ok = np.isfinite(g).all(-1) & (v >= 0.5)
    ringkas.append(dict(rekaman=nama, n32=len(g), frac_langkah_ada_pose=float(np.isfinite(g[:, 7, 0]).mean()),
                        frac_sendi_valid=float(ok.mean()), crop_w=cw, crop_h=ch))
df = pd.DataFrame(ringkas); df.to_csv(BASE / "ringkasan_cache_pose.csv", index=False)
print("Tersimpan:", len(df), "| tidak ditemukan:", hilang)
print(df.describe().round(3))

**Konteks:** Memeriksa ketersediaan GPU dan versi PyTorch.

In [ ]:
# Cek GPU
import torch; print(torch.cuda.is_available(), torch.__version__)

**Konteks:** Menulis `models/stgcn.py`: ST-GCN pada 17 keypoint 2D dengan tiga skema normalisasi (piksel, pinggul, skala) dan jendela 128 sampel.

In [ ]:
%%writefile /content/fog-benchamark-starter/models/stgcn.py
"""ST-GCN untuk FoG dari keypoint 2D (Yan dkk., 2018; Filtjens dkk., 2022)."""
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from fogbench import data as D
from models.base import FoGModel

FS = 32
POSE_DIR = Path(os.environ.get("FOG_POSE_DIR", "pose_grid32"))
SKEMA = os.environ.get("FOG_STGCN_SKEMA", "pinggul")
EPOCHS = int(os.environ.get("FOG_STGCN_EPOCHS", "30"))
WIN, STRIDE, BATCH, LR, WD = 128, 32, 32, 5e-4, 1e-4
MIN_VIS = 0.5
V = 17
EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (3, 5), (2, 4), (4, 6), (1, 7), (2, 8), (7, 8), (7, 9), (9, 11),
         (8, 10), (10, 12), (11, 13), (11, 15), (13, 15), (12, 14), (12, 16), (14, 16)]
HIP = (7, 8)
_CACHE: dict = {}


def _muat(subject, session):
    key = (subject, session)
    if key not in _CACHE:
        d = np.load(POSE_DIR / f"PDFE{subject:02d}_{session + 1}.npz")
        _CACHE[key] = (d["xy"].astype(np.float32), d["vis"].astype(np.float32))
    return _CACHE[key]


def _selaraskan(a, n):
    if len(a) >= n:
        return a[:n]
    pad = np.full((n - len(a),) + a.shape[1:], np.nan, np.float32)
    return np.concatenate([a, pad])


def _skala_tubuh(xy):
    """Median panjang torso; cadangan pinggul-ankle."""
    with np.errstate(all="ignore"):
        sh, hp, an = xy[:, 1:3].mean(1), xy[:, 7:9].mean(1), xy[:, 11:13].mean(1)
        s = np.nanmedian(np.linalg.norm(sh - hp, axis=1))
        if not np.isfinite(s) or s < 1:
            s = np.nanmedian(np.linalg.norm(hp - an, axis=1))
    return float(s) if np.isfinite(s) and s >= 1 else 1.0


def _praproses(xy, vis, skema):
    """xy (T,17,2) piksel -> (koordinat, valid (T,17))."""
    valid = np.isfinite(xy).all(-1) & (np.nan_to_num(vis) >= MIN_VIS)
    X = xy.copy()
    if skema in ("pinggul", "skala"):
        hip = X[:, HIP, :].mean(1)
        X = X - hip[:, None, :]
        valid &= np.isfinite(hip).all(-1)[:, None]
        if skema == "skala":
            X = X / _skala_tubuh(xy)
    return X, valid


def _ke_tensor(X, valid, mu, sd):
    Z = (X - mu) / sd
    Z = np.where(valid[..., None], Z, 0.0).astype(np.float32)
    return np.concatenate([Z, valid[..., None].astype(np.float32)], axis=-1)      # (T,17,3)


def bangun_A():
    """A (3,V,V): akar, sentripetal, sentrifugal."""
    adj = np.zeros((V, V))
    for i, j in EDGES:
        adj[i, j] = adj[j, i] = 1
    hop = np.full(V, np.inf)
    hop[list(HIP)] = 0
    for _ in range(V):
        for i, j in EDGES:
            hop[i], hop[j] = min(hop[i], hop[j] + 1), min(hop[j], hop[i] + 1)
    A = np.zeros((3, V, V), np.float32)
    A[0] = np.eye(V)
    for i in range(V):
        for j in range(V):
            if adj[i, j]:
                A[1 if hop[j] < hop[i] else 2 if hop[j] > hop[i] else 0][j, i] = 1
    deg = (A.sum(0)).sum(0) + 1e-6
    return torch.tensor(A / deg[None, None, :], dtype=torch.float32)


class BlokSTGCN(nn.Module):
    def __init__(self, cin, cout, A, dil=1, k_t=9, drop=0.25):
        super().__init__()
        self.register_buffer("A", A)
        self.M = nn.Parameter(torch.ones_like(A))
        self.gc = nn.Conv2d(cin, cout * A.shape[0], 1)
        self.bn1 = nn.BatchNorm2d(cout)
        pad = (k_t - 1) // 2 * dil
        self.tc = nn.Sequential(nn.Conv2d(cout, cout, (k_t, 1), padding=(pad, 0), dilation=(dil, 1)),
                                nn.BatchNorm2d(cout), nn.Dropout(drop))
        self.res = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout))
        self.relu = nn.ReLU()
        self.K = A.shape[0]

    def forward(self, x):                                             # x: (N,C,T,V)
        n, _, t, v = x.shape
        y = self.gc(x).view(n, self.K, -1, t, v)
        y = torch.einsum("nkctv,kvw->nctw", y, self.A * self.M)
        y = self.relu(self.bn1(y))
        return self.relu(self.tc(y) + self.res(x))


class STGCN(nn.Module):
    """4 blok, dilasi (1,1,2,4), medan reseptif ~2 s; logit per sampel."""
    def __init__(self, cin=3, ch=(32, 32, 64, 64), dil=(1, 1, 2, 4)):
        super().__init__()
        A = bangun_A()
        self.bn0 = nn.BatchNorm1d(cin * V)
        blok, c = [], cin
        for co, d in zip(ch, dil):
            blok.append(BlokSTGCN(c, co, A, d)); c = co
        self.blok = nn.Sequential(*blok)
        self.head = nn.Conv1d(c, 1, 1)

    def forward(self, x):                                             # x: (N,T,V,C)
        n, t, v, c = x.shape
        x = x.permute(0, 3, 2, 1).reshape(n, c * v, t)
        x = self.bn0(x).view(n, c, v, t).permute(0, 1, 3, 2)          # (N,C,T,V)
        x = self.blok(x).mean(-1)                                     # pool sendi -> (N,C,T)
        return self.head(x).squeeze(1)                                # (N,T)


class STGCNFoG(FoGModel):
    def __init__(self, seed=0, skema=SKEMA, epochs=EPOCHS):
        self.seed, self.skema, self.epochs = seed, skema, epochs
        self.name = f"stgcn-{skema}"
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.net, self.mu, self.sd = None, None, None
        self.riwayat = []

    def _rekaman(self, r):
        n32 = len(D.to_eval_rate(r.y, r.fs, FS, is_label=True))
        xy, vis = _muat(r.subject, r.session)
        return _selaraskan(xy, n32), _selaraskan(vis, n32), n32

    def fit(self, recordings):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        data = []
        for r in recordings:
            xy, vis, n32 = self._rekaman(r)
            X, valid = _praproses(xy, vis, self.skema)
            y = np.asarray(D.to_eval_rate(r.y, r.fs, FS, is_label=True)).astype(np.float32)
            data.append((X, valid, y))
        # statistik normalisasi: hanya dari pasien latih
        allx = np.concatenate([X[v] for X, v, _ in data])
        self.mu = (allx.mean(0) if self.skema == "piksel" else np.zeros(2)).astype(np.float32)
        self.sd = np.maximum(allx.std(0), 1e-6).astype(np.float32)
        T = []
        for X, valid, y in data:
            Z = _ke_tensor(X, valid, self.mu, self.sd)
            for s in range(0, max(len(y) - WIN, 0) + 1, STRIDE):
                if valid[s:s + WIN].any(-1).mean() >= 0.2 and len(y[s:s + WIN]) == WIN:
                    T.append((Z[s:s + WIN], y[s:s + WIN]))
        Xw = torch.tensor(np.stack([a for a, _ in T])); yw = torch.tensor(np.stack([b for _, b in T]))
        pos = float(yw.sum()); pw = (yw.numel() - pos) / max(pos, 1.0)
        self.net = STGCN().to(self.dev)
        opt = torch.optim.Adam(self.net.parameters(), lr=LR, weight_decay=WD)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=self.dev))
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.net.train(); perm = torch.randperm(len(Xw), generator=g); tot = 0.0
            for i in range(0, len(perm), BATCH):
                idx = perm[i:i + BATCH]
                xb = Xw[idx].to(self.dev); yb = yw[idx].to(self.dev)
                xb = xb + 0.02 * torch.randn_like(xb) * (xb[..., 2:3] > 0)
                loss = loss_fn(self.net(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
            self.riwayat.append(tot / len(Xw))
            print(f"  epoch {ep + 1}/{self.epochs} loss {self.riwayat[-1]:.4f}", flush=True)
        self.n_jendela, self.pos_weight = len(Xw), pw

    @torch.no_grad()
    def predict_proba(self, recording):
        n128 = len(recording.y)
        xy, vis, n32 = self._rekaman(recording)
        X, valid = _praproses(xy, vis, self.skema)
        Z = torch.tensor(_ke_tensor(X, valid, self.mu, self.sd))[None].to(self.dev)
        self.net.eval()
        p32 = torch.sigmoid(self.net(Z))[0].cpu().numpy()
        p = np.repeat(p32, recording.fs // FS)
        if len(p) < n128:
            p = np.concatenate([p, np.full(n128 - len(p), p[-1])])
        return p[:n128].astype(np.float32)

    def card(self):
        npar = sum(p.numel() for p in self.net.parameters()) if self.net is not None else None
        return {
            "nama": self.name, "n_parameter": npar,
            "hiperparameter": {"skema_normalisasi": self.skema, "sendi": V, "kanal_masukan": 3,
                               "blok": 4, "kanal": [32, 32, 64, 64], "kernel_temporal": 9, "dilasi": [1, 1, 2, 4],
                               "partisi": "spatial configuration (akar, sentripetal, sentrifugal)",
                               "jendela_latih": WIN, "langkah_jendela": STRIDE, "batch": BATCH, "lr": LR,
                               "weight_decay": WD, "epoch": self.epochs, "optimizer": "Adam",
                               "loss": "BCE berbobot (pos_weight = n_neg/n_pos pasien latih)",
                               "min_visibility": MIN_VIS, "seed": self.seed},
            "deviasi": [
                "Masukan MediaPipe BlazePose 2D, subset 17 sendi; Yan dkk. (2018) 18 sendi OpenPose, Filtjens dkk. (2022) 25 titik 3D Kinect.",
                "Jaringan jauh lebih kecil (4 blok, 32-64 kanal) dibanding Yan dkk. (9 blok, 64-256 kanal); tanpa tahap penghalusan MS-GCN (setara ST-GCN satu tahap).",
                "Kernel temporal 9 seperti Yan dkk., dengan dilasi (1,1,2,4) tanpa stride; keluaran klasifikasi per sampel (segmentasi), bukan per klip.",
                "Loss: BCE berbobot saja; tanpa truncated MSE (tau=4, lambda=0,15) milik Filtjens dkk.",
                "Adam lr 5e-4 (sama dengan Filtjens dkk.), tetapi 30 epoch (Filtjens 100), batch 32 (Filtjens 16); tanpa penjadwalan lr dan tanpa early stopping.",
                "Kanal masukan ketiga = penanda sendi valid (visibility >= 0,5 dan koordinat terhingga); sendi tidak valid diisi 0 setelah normalisasi.",
                "Statistik normalisasi dari pasien latih; skema 'skala' memakai median panjang torso per rekaman (hanya dari pose rekaman itu sendiri, tanpa label).",
                "Augmentasi hanya derau Gaussian kecil; tanpa flip kiri-kanan atau pemotongan acak.",
                "Tanpa penghalusan probabilitas keluaran (model XGBoost Li memakai rerata bergerak 1 detik).",
                "Dataset berbeda (Figshare, label FoG dari IMU, pasien Parkinson) vs. NTU-RGB+D/Kinetics (Yan) dan Kinect (Filtjens)."]}


def factory(seed: int = 0) -> STGCNFoG:
    return STGCNFoG(seed=seed, skema=os.environ.get("FOG_STGCN_SKEMA", "pinggul"))

**Konteks:** Uji ST-GCN pada satu fold (subjek 4, seed 0, skema pinggul) untuk memastikan pelatihan dan penyimpanan hasil berjalan.

In [ ]:
# Uji ST-GCN pada 1 fold (subjek 4, seed 0)
import time
os.environ["FOG_POSE_DIR"] = str(POSE_GRID_DIR)
SKEMA = "pinggul"                       # piksel | pinggul | skala
os.environ["FOG_STGCN_SKEMA"] = SKEMA
os.environ["FOG_STGCN_EPOCHS"] = "30"
import models.stgcn as S; importlib.reload(S)

OUT_S = HASIL_FOGBENCH / f"stgcn-{SKEMA}_figshare"
t0 = time.time()
rec, m_mean, m_seed, masalah, model_t = satu_fold(S.factory, recs, 4, OUT_S, f"stgcn-{SKEMA}", "figshare", seeds=[0])
print(f"Selesai dalam {time.time() - t0:.0f} s")
print({k: round(v, 3) for k, v in m_mean.items() if isinstance(v, (int, float))})
print("Masalah validasi:", masalah, "| parameter:", rec["model_card"]["n_parameter"], "| ambang:", round(rec["threshold"], 3))

**Konteks:** Menjalankan ST-GCN 3 seed (skema pinggul) dan skema piksel/skala (seed 0) pada fold uji, bila saklar `RUN_STGCN_3SEED_DAN_SKEMA` aktif.

In [ ]:
# ST-GCN 3 seed (pinggul) + uji skema piksel dan skala
if RUN_STGCN_3SEED_DAN_SKEMA:
    os.environ["FOG_POSE_DIR"] = str(POSE_GRID_DIR)
    os.environ["FOG_STGCN_EPOCHS"] = "30"
    import models.stgcn as S; importlib.reload(S)
    RUN = [("pinggul", [0, 1, 2]), ("piksel", [0]), ("skala", [0])]
    for SKEMA, seeds in RUN:
        os.environ["FOG_STGCN_SKEMA"] = SKEMA
        OUT_S = HASIL_FOGBENCH / f"stgcn-{SKEMA}_figshare"
        t0 = time.time()
        rec, m_mean, m_seed, masalah, _ = satu_fold(S.factory, recs, 4, OUT_S, f"stgcn-{SKEMA}", "figshare", seeds=seeds)
        print(f"\n=== {SKEMA} | seed {seeds} | {time.time() - t0:.0f} s | validasi: {masalah} ===")
        for sd, m in zip(seeds, m_seed):
            print(f"  seed {sd}: AUROC {m['auroc']:.3f} AUPRC {m['auprc']:.3f} sens {m['sensitivitas']:.3f} "
                  f"spes {m['spesifisitas']:.3f} F1 {m['f1']:.3f} F1-episode {m['episode_f1']:.3f}")
        print("  p_mean:", {k: round(float(m_mean[k]), 3) for k in ("auroc", "auprc", "f1", "episode_f1")}, "| ambang", round(rec["threshold"], 3))
else:
    print("RUN_STGCN_3SEED_DAN_SKEMA=False -> dilewati.")

**Konteks:** Menyalin berkas model Li dan ST-GCN dari Colab ke folder kode di Drive sebagai cadangan.

In [ ]:
# Salin berkas model ke Drive
import shutil
for f in ["li_kinematik_xgb.py", "stgcn.py"]:
    shutil.copy(f"/content/fog-benchamark-starter/models/{f}", KODE_DIR / f)
print(sorted(p.name for p in KODE_DIR.iterdir()))

**Konteks:** Mencantumkan berkas hasil per fold dan CSV Week 2 beserta ukurannya, sebagai pemeriksaan akhir.

In [ ]:
# Daftar berkas hasil Week 2
for d in sorted(HASIL_FOGBENCH.glob("*_figshare")):
    print(d.name, "->", sorted(p.name for p in d.glob("*.json")))
print()
for p in sorted(BASE.glob("*.csv")):
    print(f"{p.name:45s} {p.stat().st_size/1024:8.1f} KB")